In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import (Dense, LSTM, Bidirectional, Dropout,
                                     GlobalAveragePooling2D, Input,
                                     Concatenate, Reshape)
from tensorflow.keras.models import Model
import matplotlib.pyplot as plt

    Dataset link: https://www.kaggle.com/datasets/afridirahman/brain-stroke-ct-image-dataset?select=Brain_Data_Organised

In [ ]:
data = []
labels = []

dataset_path = "/kaggle/input/datasets/afridirahman/brain-stroke-ct-image-dataset/Brain_Data_Organised"

categories = ["Normal", "Stroke"]   # IMPORTANT (check folder names exactly)

for category in categories:
    path = os.path.join(dataset_path, category)
    label = 1 if category.lower() == "stroke" else 0

    for img_name in os.listdir(path):
        img_path = os.path.join(path, img_name)

        try:
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, (128,128))  # initial resize
            data.append(img)
            labels.append(label)
        except:
            pass

data = np.array(data)
labels = np.array(labels)

print("Data shape:", data.shape)
print("Labels shape:", labels.shape)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    data, labels, test_size=0.2, random_state=42
)

In [ ]:
# Resize to 224 for ResNet
x_train = np.array([cv2.resize(img, (224,224)) for img in x_train])
x_test  = np.array([cv2.resize(img, (224,224)) for img in x_test])

# Convert grayscale → RGB
x_train = np.stack((x_train,)*3, axis=-1)
x_test  = np.stack((x_test,)*3, axis=-1)

# Normalize
x_train = x_train / 255.0
x_test  = x_test / 255.0

print("Train shape:", x_train.shape)

In [ ]:
from sklearn.utils import class_weight
import numpy as np

class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))

print(class_weights)

In [ ]:
input_layer = Input(shape=(224,224,3))

# CNN
base_model = ResNet50(
    weights='imagenet',   # Kaggle supports internet
    include_top=False,
    input_tensor=input_layer
)

# Freeze layers
for layer in base_model.layers:
    layer.trainable = False

# Feature extraction
x = base_model.output
x = GlobalAveragePooling2D()(x)

# -------- BiLSTM --------
lstm_input = Reshape((1, x.shape[-1]))(x)
lstm_branch = Bidirectional(LSTM(128))(lstm_input)

# -------- DNN --------
dnn_branch = Dense(256, activation='relu')(x)
dnn_branch = Dropout(0.5)(dnn_branch)
dnn_branch = Dense(128, activation='relu')(dnn_branch)

# -------- Combine --------
combined = Concatenate()([lstm_branch, dnn_branch])

z = Dense(128, activation='relu')(combined)
z = Dropout(0.5)(z)
output = Dense(1, activation='sigmoid')(z)

model11 = Model(inputs=input_layer, outputs=output)

model11.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model11.summary()

In [ ]:
history = model11.fit(
    x_train, y_train,
    validation_data=(x_test, y_test),
    epochs=300,
    batch_size=16,
    class_weight=class_weights 
)


In [ ]:
model11.save("/kaggle/working/stroke_model12.h5")

In [ ]:
import os
print(os.listdir('/kaggle/working'))

In [ ]:
import shutil

shutil.copy(
    '/kaggle/working/stroke_model12.h5',
    '/kaggle/working/final_model.h5'
)

print("Model ready for download!")

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Input
from tensorflow.keras.models import Model

input_layer = Input(shape=(224,224,3))

base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_tensor=input_layer
)

# Fine-tuning
for layer in base_model.layers[:-30]:
    layer.trainable = False

for layer in base_model.layers[-30:]:
    layer.trainable = True

x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)

x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)

output = Dense(1, activation='sigmoid')(x)

model123 = Model(inputs=input_layer, outputs=output)

model123.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model123.summary()

## from tensorflow.keras.applications import DenseNet121

input_layer = Input(shape=(224,224,3))

base_model = DenseNet121(
    weights='imagenet',
    include_top=False,
    input_tensor=input_layer
)

# Fine-tune
for layer in base_model.layers[:-40]:
    layer.trainable = False

for layer in base_model.layers[-40:]:
    layer.trainable = True

x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)

x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)

output = Dense(1, activation='sigmoid')(x)

model1223 = Model(inputs=input_layer, outputs=output)

model123.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils import class_weight
import numpy as np

# Class weights
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights))

# Callbacks
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ReduceLROnPlateau(patience=3, factor=0.3)
]

# # Train
# history = model.fit(
#     x_train, y_train,
#     validation_data=(x_test, y_test),
#     epochs=25,
#     batch_size=16,
#     class_weight=class_weights,
#     callbacks=callbacks
# )
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

history = model123.fit(
    datagen.flow(x_train, y_train, batch_size=16),
    validation_data=(x_test, y_test),
    epochs=100,
    class_weight=class_weights
)

In [ ]:
import numpy as np

print("Class distribution:", np.bincount(labels))

In [ ]:
model123.save("stroke_model.h5")

In [ ]:
loss, acc = model11.evaluate(x_test, y_test)
print("Test Accuracy:", acc)

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title("Accuracy")
plt.legend(["Train", "Validation"])
plt.show()

In [ ]:
def predict_image(img_path):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (224,224))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    pred = model123.predict(img)

    if pred[0][0] > 0.5:
        print("🧠 Stroke Detected")
    else:
        print("✅ Normal Brain")
predict_image('/kaggle/input/datasets/afridirahman/brain-stroke-ct-image-dataset/Brain_Data_Organised/Stroke/58 (1).jpg')